In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T
from torch.utils.data import DataLoader, Subset
import numpy as np
from tqdm import tqdm


In [2]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

BASELINE_PATH = "/kaggle/input/baseline-75-dbc/densenet_bc100_baseline (1).pt"
PRUNED_PATH   = "densenet_bc100_structural_pruned.pt"

PRUNE_RATIO = 0.1       
CALIB_SAMPLES = 1000

MEAN = [0.5071, 0.4867, 0.4408]
STD  = [0.2675, 0.2565, 0.2761]

print("Device:", DEVICE)


Device: cuda


In [3]:
transform = T.Compose([
    T.ToTensor(),
    T.Normalize(MEAN, STD)
])

full_train = torchvision.datasets.CIFAR100(
    root="./data", train=True, download=True, transform=transform
)

idx = torch.randperm(len(full_train))[:CALIB_SAMPLES]
calib_set = Subset(full_train, idx)

calib_loader = DataLoader(
    calib_set, batch_size=64, shuffle=False, num_workers=2
)

print("Calibration samples:", len(calib_set))


100%|██████████| 169M/169M [00:01<00:00, 103MB/s]


Calibration samples: 1000


In [4]:
class Bottleneck(nn.Module):
    def __init__(self, in_planes, growth_rate):
        super().__init__()
        inter = 4 * growth_rate
        self.bn1 = nn.BatchNorm2d(in_planes)
        self.conv1 = nn.Conv2d(in_planes, inter, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(inter)
        self.conv2 = nn.Conv2d(inter, growth_rate, 3, padding=1, bias=False)

    def forward(self, x):
        out = self.conv1(F.relu(self.bn1(x)))
        out = self.conv2(F.relu(self.bn2(out)))
        return torch.cat([x, out], 1)


In [5]:
class Transition(nn.Module):
    def __init__(self, in_planes, out_planes):
        super().__init__()
        self.bn = nn.BatchNorm2d(in_planes)
        self.conv = nn.Conv2d(in_planes, out_planes, 1, bias=False)

    def forward(self, x):
        return F.avg_pool2d(self.conv(F.relu(self.bn(x))), 2)


In [6]:
class DenseNetBC(nn.Module):
    def __init__(self, depth=100, growth_rate=12, reduction=0.5, num_classes=100):
        super().__init__()
        n = (depth - 4) // 6
        c = 2 * growth_rate

        self.conv1 = nn.Conv2d(3, c, 3, padding=1, bias=False)

        self.dense1 = self._make_dense(c, n, growth_rate)
        c += n * growth_rate
        self.trans1 = Transition(c, int(c * reduction))
        c = int(c * reduction)

        self.dense2 = self._make_dense(c, n, growth_rate)
        c += n * growth_rate
        self.trans2 = Transition(c, int(c * reduction))
        c = int(c * reduction)

        self.dense3 = self._make_dense(c, n, growth_rate)
        c += n * growth_rate

        self.bn = nn.BatchNorm2d(c)
        self.fc = nn.Linear(c, num_classes)

    def _make_dense(self, c, n, g):
        layers = []
        for _ in range(n):
            layers.append(Bottleneck(c, g))
            c += g
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.conv1(x)
        x = self.trans1(self.dense1(x))
        x = self.trans2(self.dense2(x))
        x = self.dense3(x)
        x = F.relu(self.bn(x))
        x = F.adaptive_avg_pool2d(x, 1)
        return self.fc(x.view(x.size(0), -1))


In [7]:
model = DenseNetBC().to(DEVICE)
model.load_state_dict(torch.load(BASELINE_PATH, map_location=DEVICE))
model.eval()

print("✅ Baseline loaded")


✅ Baseline loaded


In [8]:
activations, gradients = {}, {}

def fwd_hook(name):
    def hook(m, i, o): activations[name] = o.detach()
    return hook

def bwd_hook(name):
    def hook(m, gi, go): gradients[name] = go[0].detach()
    return hook

for name, m in model.named_modules():
    if isinstance(m, nn.Conv2d) and name.endswith("conv2"):
        m.register_forward_hook(fwd_hook(name))
        m.register_full_backward_hook(bwd_hook(name))


In [9]:
criterion = nn.CrossEntropyLoss()
influence = {}

for imgs, labels in tqdm(calib_loader, desc="Influence"):
    imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
    out = model(imgs)
    loss = criterion(out, labels)
    model.zero_grad()
    loss.backward()

    for k in activations:
        A, G = activations[k], gradients[k]
        score = torch.abs(A * G).sum((2,3))
        influence.setdefault(k, []).append(score.cpu())


Influence: 100%|██████████| 16/16 [00:03<00:00,  4.07it/s]


In [10]:
keep_idx = {}
for k, v in influence.items():
    v = torch.cat(v, 0)
    med = v.median(0).values
    keep = int(med.numel() * (1 - PRUNE_RATIO))
    keep_idx[k] = torch.topk(med, keep).indices


In [11]:
class PrunedBottleneck(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        inter = 4 * out_c
        self.bn1 = nn.BatchNorm2d(in_c)
        self.conv1 = nn.Conv2d(in_c, inter, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(inter)
        self.conv2 = nn.Conv2d(inter, out_c, 3, padding=1, bias=False)

    def forward(self, x):
        out = self.conv1(F.relu(self.bn1(x)))
        out = self.conv2(F.relu(self.bn2(out)))
        return torch.cat([x, out], 1)


In [12]:
class PrunedDenseNetBC(nn.Module):
    def __init__(self, keep_cfg, reduction=0.5, num_classes=100):
        super().__init__()
        self.ptr = 0
        n = len(keep_cfg)//3

        c = 2 * len(keep_cfg[0])
        self.conv1 = nn.Conv2d(3, c, 3, padding=1, bias=False)

        self.dense1 = self._make_dense(c, n, keep_cfg)
        c += sum(len(keep_cfg[i]) for i in range(n))
        self.trans1 = Transition(c, int(c*reduction))
        c = int(c*reduction)

        self.dense2 = self._make_dense(c, n, keep_cfg)
        c += sum(len(keep_cfg[i]) for i in range(n,2*n))
        self.trans2 = Transition(c, int(c*reduction))
        c = int(c*reduction)

        self.dense3 = self._make_dense(c, n, keep_cfg)
        c += sum(len(keep_cfg[i]) for i in range(2*n,3*n))

        self.bn = nn.BatchNorm2d(c)
        self.fc = nn.Linear(c, num_classes)

    def _make_dense(self, c, n, cfg):
        layers=[]
        for _ in range(n):
            k = len(cfg[self.ptr])
            layers.append(PrunedBottleneck(c, k))
            c += k
            self.ptr += 1
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.conv1(x)
        x = self.trans1(self.dense1(x))
        x = self.trans2(self.dense2(x))
        x = self.dense3(x)
        x = F.relu(self.bn(x))
        x = F.adaptive_avg_pool2d(x, 1)
        return self.fc(x.view(x.size(0), -1))


In [13]:
cfg = [keep_idx[k] for k in sorted(keep_idx)]
pruned = PrunedDenseNetBC(cfg).to(DEVICE)

torch.save(pruned.state_dict(), PRUNED_PATH)

def count(m): return sum(p.numel() for p in m.parameters())

print("\n===== PARAM COUNTS =====")
print("Baseline:", count(model))
print("Pruned  :", count(pruned))
print("Reduction:", (1 - count(pruned)/count(model))*100, "%")



===== PARAM COUNTS =====
Baseline: 800032
Pruned  : 563780
Reduction: 29.53031878724851 %


## Finetuning

In [14]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T
from torch.utils.data import DataLoader
from tqdm import tqdm


In [15]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# paths
PRUNED_MODEL_PATH = "densenet_bc100_structural_pruned.pt"
SAVE_BEST_PATH   = "IAFP_densenet_bc100_structural_pruned.pt"

BATCH_SIZE = 128
EPOCHS = 50                # usually 10–20 is enough
BASE_LR = 0.03              # important
WEIGHT_DECAY = 1e-4

MEAN = [0.5071, 0.4867, 0.4408]
STD  = [0.2675, 0.2565, 0.2761]

print("Device:", DEVICE)


Device: cuda


In [16]:
train_transform = T.Compose([
    T.RandomCrop(32, padding=4),
    T.RandomHorizontalFlip(),
    T.ToTensor(),
    T.Normalize(MEAN, STD)
])

test_transform = T.Compose([
    T.ToTensor(),
    T.Normalize(MEAN, STD)
])

train_set = torchvision.datasets.CIFAR100(
    root="./data", train=True, download=True, transform=train_transform
)

test_set = torchvision.datasets.CIFAR100(
    root="./data", train=False, download=True, transform=test_transform
)

train_loader = DataLoader(
    train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=2
)

test_loader = DataLoader(
    test_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=2
)

print("✅ CIFAR-100 loaders ready")


✅ CIFAR-100 loaders ready


In [17]:
pruned = PrunedDenseNetBC(cfg).to(DEVICE)
pruned.load_state_dict(torch.load(PRUNED_MODEL_PATH, map_location=DEVICE))
pruned.train()

print("✅ Structurally pruned model loaded for fine-tuning")


✅ Structurally pruned model loaded for fine-tuning


In [18]:
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

optimizer = torch.optim.SGD(
    pruned.parameters(),
    lr=BASE_LR,
    momentum=0.9,
    weight_decay=WEIGHT_DECAY
)

scheduler = torch.optim.lr_scheduler.MultiStepLR(
    optimizer,
    milestones=[8, 12],   # staged decay
    gamma=0.1
)

print("✅ Optimizer & scheduler ready")


✅ Optimizer & scheduler ready


In [19]:
def evaluate_accuracy(model, loader):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            out = model(x)
            pred = out.argmax(dim=1)
            correct += (pred == y).sum().item()
            total += y.size(0)

    model.train()
    return correct / total


In [20]:
best_acc = 0.0

for epoch in range(EPOCHS):
    running_loss = 0.0

    for x, y in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
        x, y = x.to(DEVICE), y.to(DEVICE)

        optimizer.zero_grad()
        out = pruned(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    scheduler.step()

    acc = evaluate_accuracy(pruned, test_loader)

    print(f"\nEpoch {epoch+1}")
    print(f"Train Loss: {running_loss/len(train_loader):.4f}")
    print(f"Test Acc  : {acc*100:.2f}%")

    if acc > best_acc:
        best_acc = acc
        torch.save(pruned.state_dict(), SAVE_BEST_PATH)
        print("✅ Best fine-tuned model saved")


Epoch 1/50: 100%|██████████| 391/391 [01:58<00:00,  3.29it/s]



Epoch 1
Train Loss: 4.0245
Test Acc  : 13.70%
✅ Best fine-tuned model saved


Epoch 2/50: 100%|██████████| 391/391 [02:05<00:00,  3.11it/s]



Epoch 2
Train Loss: 3.4764
Test Acc  : 23.20%
✅ Best fine-tuned model saved


Epoch 3/50: 100%|██████████| 391/391 [02:06<00:00,  3.09it/s]


...


Epoch 48
Train Loss: 1.7977
Test Acc  : 62.35%
✅ Best fine-tuned model saved


Epoch 49/50: 100%|██████████| 391/391 [02:06<00:00,  3.10it/s]



Epoch 49
Train Loss: 1.7966
Test Acc  : 62.11%


Epoch 50/50: 100%|██████████| 391/391 [02:06<00:00,  3.10it/s]



Epoch 50
Train Loss: 1.7959
Test Acc  : 61.95%


In [21]:
pruned.load_state_dict(torch.load(SAVE_BEST_PATH, map_location=DEVICE))
final_acc = evaluate_accuracy(pruned, test_loader)

print("\n====== FINAL FINE-TUNED RESULT ======")
print(f"Accuracy: {final_acc*100:.2f}%")
print("===================================")



====== FINAL FINE-TUNED RESULT ======
Accuracy: 62.35%
